#Musical Artist Recommendation

In [ ]:
"""
Mount Drive and Import Libraries
"""

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.sparse import csr_matrix, coo_matrix
!pip install implicit
import implicit

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### "Business" Understanding

What is the problem that we are trying to solve?

I am an avid music listener. I listen to music while I commute, exercise, work, game, and relax, however playing the same artists and songs over and over again can get quite boring. In past years I would spend hours searching for new music to enjoy and creating playlists, but nowadays I hardly have the time to explore the depths of spotify or soundcloud due to my busy schedule. The goal of this project is to build a ML system that can recommend new artists to me (or another user) based on the artists who I have liked in the past. That way I can passively listen to these recommended artists, saving me hours of searching and keeping my playlists from getting too stale.







### Analytic Approach

How can we use data to answer the business question?

There are a number of ways to build a recommendation algorithm. The main two are collaborative filtering and content-based filtering.

Collaborative filtering refers to the sense that multiple users have rated the same song, collaboratively giving us a sense of what this song may be like. Here we do not have features for the items. This approach allows us to guess appropriate features for each song and predict how other users who haven't heard it may rate it in the future.

Content-Based filtering recommends items to you based on your own viewing history. It finds content with similar features to content you have already watched. Here we have features of users and items, and we want to find a way to get good matches between users and items.


To solve the business problem, we will be using the Collaborative Filtering recommendation algorithm.

### Data Requirements

What data do you need to answer the question?

In order to run the Collaborative Filtering algorithm, we need a dataset with users on one axis and artists on the other axis (User-Item Matrix). In each cell there will be a value, preferrably not binary so that we can get a finer sense of how much the user likes an artist, vs if they just listen to them or not.



### Data Collection

Where is the data sourced from, and how will you receive the data?

We will be using the last.fm dataset with users, artists, and weights which is the listen count. I downloaded this dataset from [this site](https://grouplens.org/datasets/hetrec-2011/).

We need two files - the artists file that maps artist IDs to actual artists, and the user-artists file that shows how each user rates each artist.

In [ ]:
file_path = '/content/drive/My Drive/DS Portfolio Projects/Musical Artist Recommendation Using Collaborative Filtering/hetrec2011-lastfm-2k/artists.dat'
artist_data = pd.read_csv(file_path, sep="\t")
artist_data.set_index('id', inplace=True)
artist_data.head()

,name,url,pictureURL
id,,,
1,MALICE MIZER,http://www.last.fm/music/MALICE+MIZER,http://userserve-ak.last.fm/serve/252/10808.jpg
2,Diary of Dreams,http://www.last.fm/music/Diary+of+Dreams,http://userserve-ak.last.fm/serve/252/3052066.jpg
3,Carpathian Forest,http://www.last.fm/music/Carpathian+Forest,http://userserve-ak.last.fm/serve/252/40222717...
4,Moi dix Mois,http://www.last.fm/music/Moi+dix+Mois,http://userserve-ak.last.fm/serve/252/54697835...
5,Bella Morte,http://www.last.fm/music/Bella+Morte,http://userserve-ak.last.fm/serve/252/14789013...


In [ ]:
file_path = '/content/drive/My Drive/DS Portfolio Projects/Musical Artist Recommendation Using Collaborative Filtering/hetrec2011-lastfm-2k/user_artists.dat'
user_artist_data = pd.read_csv(file_path, sep="\t")
user_artist_data.head()

,userID,artistID,weight
0,2,51,13883
1,2,52,11690
2,2,53,11351
3,2,54,10300
4,2,55,8983


### Data Understanding

Does the data we collected represent the problem to be solved?

Yes! We have a user-artist dataset which contains user-artist pairs as rows. This is great, however we need to change the structure in order to prepare the data for alternating least squares in scipy.

In [ ]:
artist_data.shape

(17632, 3)

In [ ]:
artist_data.describe()

,name,url,pictureURL
count,17632,17632,17188
unique,17632,17632,17188
top,MALICE MIZER,http://www.last.fm/music/MALICE+MIZER,http://userserve-ak.last.fm/serve/252/10808.jpg
freq,1,1,1


In [ ]:
user_artist_data.shape

(92834, 3)

In [ ]:
user_artist_data['userID'].nunique()

1892

In [ ]:
user_artist_data.describe()

,userID,artistID,weight
count,92834.000000,92834.000000,92834.00000
mean,1037.010481,3331.123145,745.24393
std,610.870436,4383.590502,3751.32208
min,2.000000,1.000000,1.00000
25%,502.000000,436.000000,107.00000
50%,1029.000000,1246.000000,260.00000
75%,1568.000000,4350.000000,614.00000
max,2100.000000,18745.000000,352698.00000


### Data Preparation

What additional work is required to manipulate and work with the data?

Let's prepare our user-artist data for alternating least squares in scipy. We also need to create a function that will pull an artist name from an artist ID, and vice versa. Once we are done with this, we will be ready to begin running collaborative filtering on our data.

In [ ]:
""" Prepare training data for scipy """

# Set indices
user_artist_data.set_index(['userID', 'artistID'], inplace=True)

# Create coo matrix
coo_matrix = scipy.sparse.coo_matrix(
    (
        user_artist_data.weight.astype(float),
        (
            user_artist_data.index.get_level_values(0),
            user_artist_data.index.get_level_values(1),
        ),
    )
)

# Convert the coo_matrix to a SciPy sparse CSR matrix
csr_matrix_data = coo_matrix.tocsr()

# Display the resulting matrix
print(csr_matrix_data)


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 92834 stored elements and shape (2101, 18746)>
  Coords	Values
  (2, 51)	13883.0
  (2, 52)	11690.0
  (2, 53)	11351.0
  (2, 54)	10300.0
  (2, 55)	8983.0
  (2, 56)	6152.0
  (2, 57)	5955.0
  (2, 58)	4616.0
  (2, 59)	4337.0
  (2, 60)	4147.0
  (2, 61)	3923.0
  (2, 62)	3782.0
  (2, 63)	3735.0
  (2, 64)	3644.0
  (2, 65)	3579.0
  (2, 66)	3312.0
  (2, 67)	3301.0
  (2, 68)	2927.0
  (2, 69)	2720.0
  (2, 70)	2686.0
  (2, 71)	2654.0
  (2, 72)	2619.0
  (2, 73)	2584.0
  (2, 74)	2547.0
  (2, 75)	2397.0
  :	:
  (2100, 8320)	284.0
  (2100, 8322)	650.0
  (2100, 8323)	456.0
  (2100, 8324)	1068.0
  (2100, 8326)	626.0
  (2100, 8327)	613.0
  (2100, 8332)	655.0
  (2100, 8344)	640.0
  (2100, 8525)	232.0
  (2100, 8529)	429.0
  (2100, 8531)	607.0
  (2100, 8533)	724.0
  (2100, 9783)	793.0
  (2100, 10008)	228.0
  (2100, 10894)	705.0
  (2100, 13677)	278.0
  (2100, 13679)	346.0
  (2100, 13978)	535.0
  (2100, 16437)	443.0
  (2100, 18725)	758.0
  (2100, 187

In [ ]:
""" Write function to pull an artist name from an ID """

def artist_id_to_name(artistID):
  return artist_data.loc[artistID,'name']

In [ ]:
artist_id_to_name(4)

'Moi dix Mois'

In [ ]:
""" Write function to pull an artist ID from a name """

def artist_name_to_id(artistName):
  try:
    return artist_data[artist_data['name'] == artistName].index[0]
  except:
    return "Artist Not Found"

In [ ]:
artist_name_to_id('A Tribe Called Quest')

619

### Modeling

When we apply collaborative filtering, do we see answers that address the business problem?

We will create user and artist matrices from the user-item matrix that when multiplied together, produce a matrix similar to the user-item matrix. This process is called Matrix Factorization and is done using Alternating Least Squares.

The steps to Alternating Least Squares are:
*   Fix user matrix and find the optimal artist matrix
*   Fix artist matrix and find the optimal user matrix

Where optimal means minmizing the least squares.



Then, we can multiply a user vector with all artist vectors to get their estimated taste/opinion for each artist. Then we can recommend n artists with the highest scores, who are new to the user.

In [ ]:
""" Define and Fit Model """
model = implicit.als.AlternatingLeastSquares(factors=50, iterations=50, regularization=0.01)
model.fit(csr_matrix_data)

  0%|          | 0/50 [00:00<?, ?it/s]

### Evaluation

Does the data model answer the initial business question, or must we adjust the data?

Nice! We now have a model that can recommend a number of artists based on a user's specific tastes! Let's get some recommendations! This will require computing my "Latent Factor".

A latent factor is a hidden, abstract variable that a model learns to represent underlying characteristics or features of users and items that aren't directly observed in the data. In collaborative filtering, latent factors might capture dimensions such as a user's preference for a specific genre, style, or mood, and an item's attributes related to those dimensions. These factors help explain why a user might prefer one item over another, even if those preferences aren't explicitly labeled in the data. Essentially, the model decomposes the user-item interaction matrix into lower-dimensional representations, and each dimension is a latent factor that encapsulates complex, underlying patterns in the data.

Taking the dot product of my latent factor with the item factors will result in recommendations for each item (song) in the database. Then I can take the highest values as the top recommendations.

In [ ]:
""" Let me create a dataset for my some of my favorite artists! """
my_artists = {
    "Michael Jackson": 500,
    "Erykah Badu": 1000,
    "Björk": 20,
    "Arctic Monkeys": 150,
    "Nelly": 200,
    "Kanye West": 500,
    "Nas": 300,
    "Marvin Gaye": 850,
    "Deftones": 1000,
    "Bob Marley": 1000,
    "Nujabes": 700,
    "Lauryn Hill": 750,
    "Sade": 950,
    "J Dilla": 450,
    "A Tribe Called Quest": 300
}
my_music_taste = pd.DataFrame({
    "artistID": [artist_name_to_id(n) for n, w in my_artists.items()],  # Replace with artistIDs
    "weight": [w for n, w in my_artists.items()]  # Higher weight means more preference
})


In [ ]:
""" Generate Recommendations for a myself! """

""" Step 1: Create a sparse vector for my music """

my_user_vector = np.zeros(csr_matrix_data.shape[1])  # Same number of artists
for _, row in my_music_taste.iterrows():
    my_user_vector[row['artistID']] = row['weight']  # Assign listening weight

""" Step 2: Compute the my Latent Factor """

# Flatten my interaction vector (my_user_vector)
r = my_user_vector.flatten()

# V: the item factors matrix from the trained model (shape: n_items x factors)
V = model.item_factors

# Retrieve regularization parameter and number of factors from the model
lambda_reg = model.regularization
factors = model.factors

# Solve for the my latent factor u by solving: (V^T V + lambda * I) u = V^T r
A = V.T.dot(V) + lambda_reg * np.eye(factors)
b = V.T.dot(r)
new_user_factor = np.linalg.solve(A, b)

""" Step 3: Score All Items Using the My Latent Factor  """

# Compute predicted preference scores for each item: score_i = u dot v_i
scores = V.dot(new_user_factor)

""" Step 4: Exclude Items Already Known """

# Extract the set of artistIDs that I already like
known_items = set(my_music_taste['artistID'])

# Build a list of (item_index, score) for items that the I have't interacted with
candidates = [(item, scores[item]) for item in range(len(scores)) if item not in known_items]

""" Step 5: Sort and Select the Top Recommendations """

# Sort candidates by predicted score in descending order and take the top 10
top_candidates = sorted(candidates, key=lambda x: x[1], reverse=True)[:10]

# Convert the item indices to artist names (assuming the index of artist_data aligns with item indices)
recommended_artist_names = [artist_data.loc[item, 'name'] for item, score in top_candidates]

# Display the recommendations
print("Recommended artists:", recommended_artist_names)


Recommended artists: ['50 Cent', 'The Roots', 'Amy Winehouse', "Lil' Wayne", '2Pac', 'Jay-Z', 'OutKast', 'Johnny Cash', 'Beck', 'Stevie Wonder']


Wow! There are many artists that I already like in my recommendations including 50 cent, lil wayne, tupac, and outkast. I will definitely be checking out the rest of them.

**How can I perform an evaluation that is not as subjective? What metrics can I use to optimize parameters in this situation?**

User feedback after deployment could be a great way to compare different models, say if we A/B test different parameters against eachother and see how key metrics are impacted. But I would like to test this model on the data we already have here. I will explore this in the next iteration of this project.

### Deployment

Can we put the model into practice?

This is currently outside of the scope for this project. In a production environment, we could run this algorithm for each user once per week/month and display the recommendations in their profile.

### Feedback

Can we get constructive feedback from the data and the stakeholder to answer the business question?

I like the recommendations I got! Some of the artists I already know and like but I got a diverse and interesting set of recommendations. Time to listen to that new music!